In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 8.8 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib


from geopy.distance import geodesic


import torch


from sklearn.preprocessing import StandardScaler, LabelEncoder


from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV


from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold, train_test_split


from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    f1_score,
    precision_score,
    recall_score,
)


from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import RandomOverSampler

from xgboost import XGBClassifier

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
!pip install kaggle

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"bianewman","key":"feb4de37ad41d44d3bec63ae62c87a8e"}'}

In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d kartik2112/fraud-detection

Dataset URL: https://www.kaggle.com/datasets/kartik2112/fraud-detection
License(s): CC0-1.0
100% 202M/202M [00:01<00:00, 140MB/s]



In [ ]:
!unzip fraud-detection.zip

Archive:  fraud-detection.zip
  inflating: fraudTest.csv           
  inflating: fraudTrain.csv          


In [ ]:
train = pd.read_csv('fraudTrain.csv', index_col = 0)
test = pd.read_csv('fraudTest.csv',index_col = 0)

In [ ]:
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

Train shape: (1296675, 22)
Test shape: (555719, 22)


In [ ]:
df_full = pd.concat([train, test], axis=0)
#df_full = build_features(df_full)
print(f"Full data shape: {df_full.shape}")

Full data shape: (1852394, 22)


In [ ]:
y_train = train["is_fraud"]
y_test  = test["is_fraud"]

print("Fraude no treino:", y_train.mean())
print("Fraude no teste: ", y_test.mean())
print("Shape treino:", train.shape)
print("Shape teste: ", test.shape)

Fraude no treino: 0.005788651743883394
Fraude no teste:  0.0038598644278853163
Shape treino: (1296675, 22)
Shape teste:  (555719, 22)


In [ ]:
df_full = pd.concat([train, test], axis=0)
#df_full = build_features(df_full)
print(f"Full data shape: {df_full.shape}")

Full data shape: (1852394, 22)


In [ ]:
def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371  # raio da Terra em km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
  #Cria features de data/hora, idade e distância geográfica
    df = df.copy()
    df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])
    df["dob"]                   = pd.to_datetime(df["dob"])

    df["hour"]        = df["trans_date_trans_time"].dt.hour
    df["day"]         = df["trans_date_trans_time"].dt.day
    df["month"]       = df["trans_date_trans_time"].dt.month
    df["day_of_week"] = df["trans_date_trans_time"].dt.dayofweek
    df["is_weekend"]  = df["day_of_week"].isin([5, 6]).astype(int)
    df["is_night"]    = df["hour"].between(0, 6).astype(int)
    df["age"]         = (df["trans_date_trans_time"] - df["dob"]).dt.days // 365

    df["distance"] = haversine_vectorized(
        df["lat"], df["long"], df["merch_lat"], df["merch_long"]
    )
    return df

DATA CLEANING

In [ ]:
# Não há nenhum valor faltante
print(f"Total de linhas duplicadas: {df_full.isnull().sum()}")

Total de linhas duplicadas: trans_date_trans_time    0
cc_num                   0
merchant                 0
category                 0
amt                      0
first                    0
last                     0
gender                   0
street                   0
city                     0
state                    0
zip                      0
lat                      0
long                     0
city_pop                 0
job                      0
dob                      0
trans_num                0
unix_time                0
merch_lat                0
merch_long               0
is_fraud                 0
dtype: int64


In [ ]:
print(f"Total de linhas duplicadas: {df_full.duplicated().sum()}")

Total de linhas duplicadas: 0


# Limpeza Inicial do dataset

In [ ]:
df = build_features(train)

In [ ]:
X = df.drop(columns=[
    "is_fraud", "trans_date_trans_time", "dob",
    "cc_num", "first", "last", "street", "zip", "unix_time", "lat", "long", "merch_lat", "trans_num","merch_long",
])
y = df["is_fraud"]

In [ ]:
X.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1296675 entries, 0 to 1296674
Data columns (total 16 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   merchant     1296675 non-null  object 
 1   category     1296675 non-null  object 
 2   amt          1296675 non-null  float64
 3   gender       1296675 non-null  object 
 4   city         1296675 non-null  object 
 5   state        1296675 non-null  object 
 6   city_pop     1296675 non-null  int64  
 7   job          1296675 non-null  object 
 8   hour         1296675 non-null  int32  
 9   day          1296675 non-null  int32  
 10  month        1296675 non-null  int32  
 11  day_of_week  1296675 non-null  int32  
 12  is_weekend   1296675 non-null  int64  
 13  is_night     1296675 non-null  int64  
 14  age          1296675 non-null  int64  
 15  distance     1296675 non-null  float64
dtypes: float64(2), int32(4), int64(4), object(6)
memory usage: 148.4+ MB


In [ ]:
# pipeline_utils Funções compartilhadas entre Random Forest e XGBoost.
#   from pipeline_utils import (
#       target_encoding_oof,
#       compute_logistic_stats,
#       apply_logistic_features,
#       preprocess_fold,
#       best_threshold,
#   )
# ─────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────
# 1. TARGET ENCODING OOF
# ─────────────────────────────────────────────────────────────
def target_encoding_oof(X_tr, y_tr, col, n_splits=5, k=100):
    """
    Retorna o array OOF encoded para X_tr e o mapping final
    (treinado no treino completo) para aplicar no val/test.
    Sem leakage: cada linha é encoded com dados que não a incluem.
    """
    global_mean   = y_tr.mean()
    train_encoded = np.zeros(len(X_tr))

    kf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42,
    )

    for fold_tr_idx, fold_val_idx in kf.split(X_tr, y_tr):

        X_fold = X_tr.iloc[fold_tr_idx]
        y_fold = y_tr.iloc[fold_tr_idx]

        temp           = X_fold[[col]].copy()
        temp["target"] = y_fold.values

        stats = temp.groupby(col)["target"].agg(["count", "mean"])
        stats["smooth"] = (
            (stats["count"] * stats["mean"] + k * global_mean)
            / (stats["count"] + k)
        )

        encoded = (
            X_tr[col]
            .iloc[fold_val_idx]
            .map(stats["smooth"])
            .fillna(global_mean)
        )
        train_encoded[fold_val_idx] = encoded.values

    # mapping final com todo o treino (para val e test)
    temp_full           = X_tr[[col]].copy()
    temp_full["target"] = y_tr.values

    stats_full = temp_full.groupby(col)["target"].agg(["count", "mean"])
    stats_full["smooth"] = (
        (stats_full["count"] * stats_full["mean"] + k * global_mean)
        / (stats_full["count"] + k)
    )

    mapping = stats_full["smooth"]

    return pd.Series(train_encoded, index=X_tr.index), mapping


# ─────────────────────────────────────────────────────────────
# 2. FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────
def compute_logistic_stats(df_tr):
    """
    Aprende todas as estatísticas de grupo apenas no treino.
    Retorna um dicionário de artefatos para uso em apply_logistic_features.
    """
    stats = {}

    stats["cat_mean"]  = df_tr.groupby("category")["amt"].mean()
    stats["cat_std"]   = df_tr.groupby("category")["amt"].std().fillna(0)
    stats["dist_mean"] = df_tr.groupby("category")["distance"].mean()
    stats["dist_std"]  = df_tr.groupby("category")["distance"].std().fillna(0)

    stats["amt_q90"]         = df_tr["amt"].quantile(0.90)
    stats["distance_median"] = df_tr["distance"].median()

    # cat_hour_mean: média de amt por (categoria × faixa horária)
    df_tmp = df_tr.copy()
    df_tmp["hour_bin"] = pd.cut(
        df_tmp["hour"],
        bins=[0, 6, 12, 18, 24],
        labels=[0, 1, 2, 3],
    )
    stats["cat_hour_mean"] = (
        df_tmp
        .groupby(["category", "hour_bin"], observed=True)["amt"]
        .mean()
    )

    return stats


def apply_logistic_features(df, stats):
    """
    Aplica features usando estatísticas externas (do treino).
    Nunca recalcula estatísticas com os dados recebidos.
    """
    df = df.copy()

    # transformações logarítmicas
    df["log_amount"]   = np.log1p(df["amt"])
    df["log_distance"] = np.log1p(df["distance"])

    # z-score e ratio por categoria
    cat_mean = df["category"].map(stats["cat_mean"])
    cat_std  = df["category"].map(stats["cat_std"])

    df["amt_zscore_cat"]     = (df["amt"] - cat_mean) / (cat_std + 1e-8)
    df["amt_ratio_category"] = df["amt"] / (cat_mean + 1e-8)

    # z-score por categoria × faixa horária
    df["hour_bin"] = pd.cut(
        df["hour"],
        bins=[0, 6, 12, 18, 24],
        labels=[0, 1, 2, 3],
    )
    idx = pd.MultiIndex.from_arrays([df["category"], df["hour_bin"]])
    cat_hour_mean_vals = (
        idx
        .map(stats["cat_hour_mean"])
        .to_series(index=df.index)
        .fillna(stats["cat_mean"].mean())
    )
    df["amt_zscore_cat_hour"] = (df["amt"] - cat_hour_mean_vals) / (cat_std + 1e-8)
    df.drop(columns=["hour_bin"], inplace=True)

    # z-score de distância por categoria
    dist_mean = df["category"].map(stats["dist_mean"])
    dist_std  = df["category"].map(stats["dist_std"])
    df["distance_zscore"] = (df["distance"] - dist_mean) / (dist_std + 1e-8)

    # risco por idade
    df["age_risk"] = (
        (df["age"] < 25).astype(int)
        + (df["age"] > 70).astype(int)
    )

    # score de risco composto
    high_amt     = (df["amt"] > stats["amt_q90"]).astype(int)
    far_distance = (df["distance"] > stats["distance_median"]).astype(int)
    df["risk_score"] = high_amt * df["is_night"] * far_distance

    # interações
    df["night_distance"] = df["is_night"] * df["log_distance"]
    df["night_amt_log"]  = df["is_night"] * df["log_amount"]
    df["amt_x_dist_log"] = df["log_amount"] * df["log_distance"]

    return df


# ─────────────────────────────────────────────────────────────
# 3. PRÉ-PROCESSAMENTO DO FOLD
# ─────────────────────────────────────────────────────────────
def preprocess_fold(X_tr, y_tr, X_vl, cat_cols, k=100):
    """
    Target encoding OOF + imputação de medianas.
    Tudo fit no treino, transform no val — sem leakage.
    """
    global_mean = y_tr.mean()

    X_tr = X_tr.copy()
    X_vl = X_vl.copy()

    # target encoding
    for col in cat_cols:
        train_enc, mapping = target_encoding_oof(X_tr, y_tr, col, k=k)
        X_tr[col] = train_enc
        X_vl[col] = X_vl[col].map(mapping).fillna(global_mean)

    # imputação de medianas
    num_cols = X_tr.select_dtypes(include=["int64", "float64"]).columns
    for col in num_cols:
        median    = X_tr[col].median()
        X_tr[col] = X_tr[col].fillna(median)
        X_vl[col] = X_vl[col].fillna(median)

    return X_tr, X_vl


# ─────────────────────────────────────────────────────────────
# 4. THRESHOLD OTIMIZADO POR F1
# ─────────────────────────────────────────────────────────────
def best_threshold(y_true, y_prob):
    """
    Retorna o threshold que maximiza o F1-Score
    sobre a curva Precision-Recall.
    """
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    f1_scores = 2 * precision * recall / (precision + recall + 1e-8)

    if len(thresholds) == 0:
        return 0.5

    best_idx = np.argmax(f1_scores[:-1])
    return float(thresholds[best_idx])

In [ ]:
# xgboost_model.py
# ─────────────────────────────────────────────────────────────
# XGBoost — Detecção de Fraudes
# sem data leakage | GPU | calibração isotônica | threshold por F1
# ─────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV

from sklearn.model_selection import (
    StratifiedKFold,
    train_test_split,
)

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
)


# ─────────────────────────────────────────────────────────────
# 1. DADOS
# ─────────────────────────────────────────────────────────────
df = build_features(train)

X = df.drop(columns=[
    "is_fraud", "trans_date_trans_time", "dob", "cc_num",
    "first", "last", "street", "zip", "unix_time",
    "lat", "long", "merch_lat", "merch_long", "trans_num",
])

y = df["is_fraud"]


# ─────────────────────────────────────────────────────────────
# 2. CONFIG
# ─────────────────────────────────────────────────────────────
cat_cols = ["merchant", "category", "gender", "job", "city", "state"]

xgb_params = {
    "n_estimators":     500,
    "learning_rate":    0.05,
    "max_depth":        6,
    "colsample_bytree": 0.8,
    "subsample":        0.8,
    "min_child_weight": 3,
    "gamma":            0.1,
    "reg_alpha":        0.5,
    "reg_lambda":       5.0,
    "max_delta_step":   2,
    "eval_metric":      "aucpr",
    "tree_method":      "hist",
    "device":           "cuda",
    "random_state":     42,
    "n_jobs":           1,
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {
    "roc_auc":   [],
    "pr_auc":    [],
    "f1":        [],
    "precision": [],
    "recall":    [],
    "accuracy":  [],
    "threshold": [],
}

threshold_acumulado = []


# ─────────────────────────────────────────────────────────────
# 3. CROSS VALIDATION
# ─────────────────────────────────────────────────────────────
for fold, (tr_idx, vl_idx) in enumerate(skf.split(X, y), 1):

    print("\n" + "=" * 50)
    print(f"Fold {fold}/5")
    print("=" * 50)

    X_tr_fold = X.iloc[tr_idx].copy()
    X_vl_fold = X.iloc[vl_idx].copy()
    y_tr_fold = y.iloc[tr_idx]
    y_vl_fold = y.iloc[vl_idx]

    # feature engineering
    logistic_stats = compute_logistic_stats(X_tr_fold)
    X_tr_fold      = apply_logistic_features(X_tr_fold, logistic_stats)
    X_vl_fold      = apply_logistic_features(X_vl_fold, logistic_stats)

    # target encoding + imputação
    X_tr_proc, X_vl_proc = preprocess_fold(
        X_tr_fold, y_tr_fold, X_vl_fold, cat_cols,
    )

    assert not any(X_tr_proc.dtypes == "object")
    assert not any(X_vl_proc.dtypes == "object")

    # scale_pos_weight
    neg = (y_tr_fold == 0).sum()
    pos = (y_tr_fold == 1).sum()
    spw = np.sqrt(neg / pos)
    print(f"scale_pos_weight: {spw:.4f}")

    params = {**xgb_params, "scale_pos_weight": spw}

    # ── split interno para threshold ─────────────────────────
    X_thr_tr, X_thr_val, y_thr_tr, y_thr_val = train_test_split(
        X_tr_proc, y_tr_fold,
        test_size=0.15,
        stratify=y_tr_fold,
        random_state=42,
    )

    # ── modelo temporário COM calibração ─────────────────────
    base_tmp  = XGBClassifier(**params)
    tmp_model = CalibratedClassifierCV(
        estimator=base_tmp,
        method="isotonic",
        cv=3,
    )
    tmp_model.fit(X_thr_tr, y_thr_tr)

    thr_fold = best_threshold(
        y_thr_val,
        tmp_model.predict_proba(X_thr_val)[:, 1],
    )
    threshold_acumulado.append(thr_fold)
    thr = np.median(threshold_acumulado)
    print(f"Threshold usado: {thr:.4f}")

    # ── modelo final COM calibração ───────────────────────────
    base_model  = XGBClassifier(**params)
    final_model = CalibratedClassifierCV(
        estimator=base_model,
        method="isotonic",
        cv=3,
    )
    final_model.fit(X_tr_proc, y_tr_fold)

    # predições
    y_prob = final_model.predict_proba(X_vl_proc)[:, 1]
    y_pred = (y_prob >= thr).astype(int)

    # métricas
    auc       = roc_auc_score(y_vl_fold, y_prob)
    pr_auc    = average_precision_score(y_vl_fold, y_prob)
    f1        = f1_score(y_vl_fold, y_pred, zero_division=0)
    precision = precision_score(y_vl_fold, y_pred, zero_division=0)
    recall    = recall_score(y_vl_fold, y_pred, zero_division=0)
    accuracy  = accuracy_score(y_vl_fold, y_pred)

    results["roc_auc"].append(auc)
    results["pr_auc"].append(pr_auc)
    results["f1"].append(f1)
    results["precision"].append(precision)
    results["recall"].append(recall)
    results["accuracy"].append(accuracy)
    results["threshold"].append(thr)

    print(
        f"AUC: {auc:.4f} | PR-AUC: {pr_auc:.4f} | "
        f"F1: {f1:.4f} | Precision: {precision:.4f} | "
        f"Recall: {recall:.4f} | Accuracy: {accuracy:.4f}"
    )


# ─────────────────────────────────────────────────────────────
# 4. RESULTADO FINAL
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 50)
print("RESULTADO FINAL — XGBOOST")
print("=" * 50)
print(f"ROC-AUC   : {np.mean(results['roc_auc']):.4f}")
print(f"PR-AUC    : {np.mean(results['pr_auc']):.4f}")
print(f"F1        : {np.mean(results['f1']):.4f}")
print(f"Precision : {np.mean(results['precision']):.4f}")
print(f"Recall    : {np.mean(results['recall']):.4f}")
print(f"Accuracy  : {np.mean(results['accuracy']):.4f}")
print(f"\nThreshold final: {np.median(threshold_acumulado):.4f}")


Fold 1/5
scale_pos_weight: 13.1052
Threshold usado: 0.3930
AUC: 0.9991 | PR-AUC: 0.9460 | F1: 0.8864 | Precision: 0.8970 | Recall: 0.8761 | Accuracy: 0.9987

Fold 2/5
scale_pos_weight: 13.1052
Threshold usado: 0.3866
AUC: 0.9987 | PR-AUC: 0.9337 | F1: 0.8760 | Precision: 0.8813 | Recall: 0.8708 | Accuracy: 0.9986

Fold 3/5
scale_pos_weight: 13.1052
Threshold usado: 0.3801
AUC: 0.9985 | PR-AUC: 0.9391 | F1: 0.8823 | Precision: 0.8907 | Recall: 0.8741 | Accuracy: 0.9987

Fold 4/5
scale_pos_weight: 13.1052
Threshold usado: 0.3676
AUC: 0.9985 | PR-AUC: 0.9434 | F1: 0.8721 | Precision: 0.8524 | Recall: 0.8927 | Accuracy: 0.9985

Fold 5/5
scale_pos_weight: 13.1063
Threshold usado: 0.3801
AUC: 0.9984 | PR-AUC: 0.9395 | F1: 0.8857 | Precision: 0.8920 | Recall: 0.8795 | Accuracy: 0.9987

RESULTADO FINAL — XGBOOST
ROC-AUC   : 0.9987
PR-AUC    : 0.9403
F1        : 0.8805
Precision : 0.8827
Recall    : 0.8786
Accuracy  : 0.9986

Threshold final: 0.3801


In [ ]:
# xgboost_final_test.py
# ─────────────────────────────────────────────────────────────
# 1. DADOS
# ─────────────────────────────────────────────────────────────
df_train = build_features(train)
df_test  = build_features(test)

drop_cols = [
    "is_fraud", "trans_date_trans_time", "dob", "cc_num",
    "first", "last", "street", "zip", "unix_time",
    "lat", "long", "merch_lat", "merch_long", "trans_num",
]

X_train = df_train.drop(columns=drop_cols)
y_train = df_train["is_fraud"]
X_test  = df_test.drop(columns=drop_cols)
y_test  = df_test["is_fraud"]

# ─────────────────────────────────────────────────────────────
# 2. PIPELINE
# ─────────────────────────────────────────────────────────────
cat_cols = ["merchant", "category", "gender", "job", "city", "state"]

logistic_stats            = compute_logistic_stats(X_train)
X_train                   = apply_logistic_features(X_train, logistic_stats)
X_test                    = apply_logistic_features(X_test,  logistic_stats)
X_train_proc, X_test_proc = preprocess_fold(X_train, y_train, X_test, cat_cols)

assert not any(X_train_proc.dtypes == "object")
assert not any(X_test_proc.dtypes == "object")

# ─────────────────────────────────────────────────────────────
# 3. HIPERPARÂMETROS
# ─────────────────────────────────────────────────────────────
spw = np.sqrt((y_train == 0).sum() / (y_train == 1).sum())

best_params_final = {
    "n_estimators":     500,
    "learning_rate":    0.05,
    "max_depth":        6,
    "colsample_bytree": 0.8,
    "subsample":        0.8,
    "min_child_weight": 3,
    "gamma":            0.1,
    "reg_alpha":        0.5,
    "reg_lambda":       5.0,
    "max_delta_step":   2,
    "scale_pos_weight": spw,
    "eval_metric":      "aucpr",
    "tree_method":      "hist",
    "device":           "cuda",
    "random_state":     42,
    "n_jobs":           1,
}

# ─────────────────────────────────────────────────────────────
# 4. MODELO FINAL COM CALIBRAÇÃO
# ─────────────────────────────────────────────────────────────
from sklearn.calibration import CalibratedClassifierCV

base_model  = XGBClassifier(**best_params_final)
final_model = CalibratedClassifierCV(
    estimator=base_model,
    method="isotonic",
    cv=3,
)
final_model.fit(X_train_proc, y_train)

# ─────────────────────────────────────────────────────────────
# 5. AVALIAÇÃO
# ─────────────────────────────────────────────────────────────
y_prob = final_model.predict_proba(X_test_proc)[:, 1]

thr_test = best_threshold(y_test, y_prob)
print(f"Threshold ótimo no teste: {thr_test:.4f}")

y_pred = (y_prob >= thr_test).astype(int)

print("\n" + "=" * 50)
print("RESULTADO FINAL — XGBOOST (TESTE)")
print("=" * 50)
print(f"ROC-AUC   : {roc_auc_score(y_test, y_prob):.4f}")
print(f"PR-AUC    : {average_precision_score(y_test, y_prob):.4f}")
print(f"F1        : {f1_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Precision : {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"\nThreshold usado: {thr_test:.4f}")

Threshold ótimo no teste: 0.1437

RESULTADO FINAL — XGBOOST (TESTE)
ROC-AUC   : 0.9713
PR-AUC    : 0.4236
F1        : 0.4098
Precision : 0.4809
Recall    : 0.3571
Accuracy  : 0.9960

Threshold usado: 0.1437
